# STAT 764 · Meeting 7 — Regularization

**Tuesday, September 29**

Least squares has no brake. Hand it enough columns and it will fit anything — including
noise — and give you back coefficients, p-values and a training R-squared that all look
like findings.

That is not a hypothetical. It is how a good share of published regressions were built,
and it is what you get from any tool that picks predictors by staring at the same data
it then reports on.

Today:

1. **A regression worth reporting** — until you find out what it was fitted to.
2. **What happens with no brake**, measured on Ames with 319 columns.
3. **What a penalty does**, and why ridge and lasso behave differently.
4. **Why the penalty cares about your units** when least squares mostly does not.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from stat764 import load

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# A 300-house sample often contains no house with a pool, so Pool_QC is entirely
# missing and the imputer (correctly) skips it -- with a warning on every fit.
# Silence that one message and nothing else.
warnings.filterwarnings("ignore", message="Skipping features without any observed values")

ames = load("ames.csv")
y = ames["SalePrice"]

# Every column except the two ID columns and the answer.
NUMERIC = [c for c in ames.select_dtypes("number").columns
           if c not in ("Order", "PID", "SalePrice")]
CATEGORICAL = ames.select_dtypes(exclude="number").columns.tolist()


def pipeline(model=None, numeric=NUMERIC, categorical=CATEGORICAL, scale=True):
    """The Meeting 2 skeleton. `scale=False` drops the StandardScaler, for Section 4."""
    num = [("fill", SimpleImputer(strategy="median"))]
    if scale:
        num.append(("scale", StandardScaler()))
    parts = [("num", Pipeline(num), numeric)]
    if categorical:
        parts.append(("cat", Pipeline([("fill", SimpleImputer(strategy="constant",
                                                              fill_value="Missing")),
                                       ("encode", OneHotEncoder(handle_unknown="ignore"))]),
                      categorical))
    return Pipeline([("prep", ColumnTransformer(parts)),
                     ("model", model or LinearRegression())])


X = ames[NUMERIC + CATEGORICAL]
width = pipeline().fit(X, y).named_steps["prep"].transform(X).shape[1]
print(f"{len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical columns "
      f"-> {width} columns once the categories are one-hot encoded")

## 1. Warm-up — would you report this model?

A hundred observations, fifty candidate predictors, and the procedure a lot of people
were taught: **forward stepwise selection**. Start with nothing; add whichever predictor
has the smallest p-value; keep going until nothing left is below 0.05.

In [ ]:
def forward_stepwise(X, y, alpha=0.05):
    """Add the predictor with the smallest p-value until none is below alpha."""
    chosen = []
    while True:
        best, best_p = None, alpha
        for c in X.columns.difference(chosen):
            p = sm.OLS(y, sm.add_constant(X[chosen + [c]])).fit().pvalues[c]
            if p < best_p:
                best, best_p = c, p
        if best is None:
            return chosen
        chosen.append(best)


def candidates(seed, n=100, p=50):
    """n observations of p candidate predictors and an outcome."""
    rng = np.random.default_rng(seed)
    X = pd.DataFrame(rng.normal(size=(n, p)), columns=[f"x{i:02d}" for i in range(p)])
    return X, pd.Series(rng.normal(size=n), name="y")


Xc, yc = candidates(7)
chosen = forward_stepwise(Xc, yc)
fit = sm.OLS(yc, sm.add_constant(Xc[chosen])).fit()
print(pd.DataFrame({"coef": fit.params[chosen], "p-value": fit.pvalues[chosen]})
      .round(4).to_string())
print(f"\n  R-squared {fit.rsquared:.3f}    overall F-test p = {fit.f_pvalue:.2g}")

Five predictors, every one significant, and an overall F-test that says the model is
nowhere near chance. Which of them would you put in the abstract?

Now read `candidates()` again. **Every column, and the outcome, is independent random
noise.** There is nothing to find. So run the whole procedure twenty times, and then take
the predictors it chose to a fresh sample — the same fifty columns, new rows.

In [ ]:
runs = []
for seed in range(20):
    Xc, yc = candidates(seed)
    chosen = forward_stepwise(Xc, yc)
    X_new, y_new = candidates(1000 + seed)                # fresh rows, same kind of noise
    again = (sm.OLS(y_new, sm.add_constant(X_new[chosen])).fit().pvalues[chosen] < 0.05
             if chosen else pd.Series(dtype=bool))
    runs.append({"chosen": len(chosen),
                 "R2": sm.OLS(yc, sm.add_constant(Xc[chosen])).fit().rsquared if chosen else 0.0,
                 "significant again": int(again.sum())})
runs = pd.DataFrame(runs)

print(f"  runs that found at least one 'significant' predictor   {(runs.chosen > 0).sum()} of 20")
print(f"  median predictors chosen {runs.chosen.median():.0f}, median R-squared {runs.R2.median():.3f}")
print(f"  chosen predictors significant again on fresh rows      "
      f"{runs['significant again'].sum()} of {runs.chosen.sum()}")
print(f"  what chance alone gives (5% of them)                   "
      f"{0.05 * runs.chosen.sum():.1f}")

**Nineteen runs out of twenty found something.** On fresh rows, the predictors they found
are significant about as often as any random column would be.

Every p-value in that first table is correct *for a predictor you had decided to test in
advance.* None of them was. Each one is the smallest of up to fifty p-values, chosen
because it was small — and a p-value was never designed to survive that. ISLP §6.1.2
describes stepwise selection; read it as the cautionary tale it is.

⚠ This is Meeting 1's lesson in a new costume. There you **scored** on the data you fit.
Here you **chose** on the data you then report. The fix is the same shape: whatever made
the choice cannot also be the evidence for it.

## 2. A model with no brake

Back to Ames, and this time give least squares **everything**: every numeric column and
every categorical one, 319 columns once encoded. Then do it twice — with the full training
set of 2,197 houses, and with only 300 of them.

Ten splits, not one. Meeting 6 taught you why.

In [ ]:
rows = []
for seed in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=733, random_state=seed)
    for n in (2197, 300):
        for name, model in [("OLS", LinearRegression()),
                            ("ridge", Ridge(alpha=10)),
                            ("lasso", Lasso(alpha=300, max_iter=20_000))]:
            fit = pipeline(model).fit(X_train[:n], y_train[:n])
            rows.append({"houses": n, "model": name,
                         "train R2": r2_score(y_train[:n], fit.predict(X_train[:n])),
                         "test R2": r2_score(y_test, fit.predict(X_test)),
                         "test MAE $": mean_absolute_error(y_test, fit.predict(X_test))})
brake = pd.DataFrame(rows)

print("  medians over 10 splits, and the worst test R-squared of the 10")
print(brake.groupby(["houses", "model"], sort=False)
      .agg(train_R2=("train R2", "median"), test_R2=("test R2", "median"),
           worst_test_R2=("test R2", "min"), test_MAE=("test MAE $", "median"))
      .round(3).to_string())

Read it in two halves.

**With 2,197 houses the brake is worth nothing.** OLS, ridge and lasso land within about
a hundredth of each other. There is enough data to pin down 319 coefficients, and a
penalty only gets in the way.

**With 300 houses, OLS falls apart.** It fits the training houses almost perfectly and
then scores 0.384 on houses it has not seen — and on its worst split, *negative*: worse
than predicting the average price for everyone. Ridge and lasso, fitted to the same 300
houses, hold at about 0.84.

⚠ **Regularization is not a better model. It is a brake, and a brake is worth exactly as
much as the data cannot pin the coefficients down.** You found that out by measuring it at
two sizes — not by assuming it.

This has a name: **overfitting** — learning noise particular to the sample as if it were
signal. You see it as training performance far above held-out performance: OLS on 300
houses scores 0.983 on the houses it was fit to and 0.384 on new ones. It depends on how
flexible the model is against how much data it has, and a penalty is the brake on it.
You have met it before without the name: Meeting 1's full tree scored 1.000 on its own
houses and 0.545 on new ones.

## 3. What the penalty does

Both methods minimize the usual squared error **plus a charge for large coefficients**:

| | minimizes | the charge |
|---|---|---|
| **ridge** | RSS + α · Σ βⱼ² | squares — big coefficients are expensive, small ones nearly free |
| **lasso** | RSS + α · Σ \|βⱼ\| | absolute values — every coefficient pays the same rate |

α is the dial. At α = 0 both are least squares. Turn it up and watch, on the same 300
houses:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=733, random_state=0)
X300, y300 = X_train[:300], y_train[:300]

dial = []
for a_ridge, a_lasso in [(0.1, 3), (1, 30), (10, 300), (100, 3000), (1000, 30000)]:
    r = pipeline(Ridge(alpha=a_ridge)).fit(X300, y300)
    l = pipeline(Lasso(alpha=a_lasso, max_iter=20_000)).fit(X300, y300)
    dial.append({"ridge α": a_ridge,
                 "ridge: size of coefficients": np.linalg.norm(r.named_steps["model"].coef_),
                 "ridge MAE $": mean_absolute_error(y_test, r.predict(X_test)),
                 "lasso α": a_lasso,
                 "lasso: nonzero of 319": int((l.named_steps["model"].coef_ != 0).sum()),
                 "lasso MAE $": mean_absolute_error(y_test, l.predict(X_test))})
print(pd.DataFrame(dial).round({"ridge: size of coefficients": 0, "ridge MAE $": 0,
                                 "lasso MAE $": 0}).to_string(index=False))

Two different behaviors from the same idea:

- **Ridge shrinks everything, and zeros nothing.** The coefficients get smaller as α
  rises, all of them together.
- **The lasso sets coefficients to exactly zero.** At the gentlest setting it keeps 230
  of 319; turned all the way up it keeps 4. That is why people use it to "select
  variables" — and Section 1 is why you should be careful saying so.

Held-out MAE is U-shaped in both: too little penalty and you are back to the collapse in
Section 2; too much and the model cannot fit anything. **Which α?** is Thursday's meeting,
and the honest answer is harder than it looks.

⚠ scikit-learn scales the two objectives differently, so ridge α = 10 and lasso α = 10
are **not** the same amount of penalty. Never compare the numbers across the two.

## 4. The penalty is measured in your units

Meeting 4 measured a leak worth exactly `0.000000`: standardizing with statistics from the
whole dataset could not change an OLS fit. Scaling a column is just a change of units, and
a least-squares coefficient absorbs it.

A penalty does not. It charges for the **size of the coefficient**, and the size of a
coefficient depends on the units of its column. Five numeric columns, the Meeting 5 split,
with and without the `StandardScaler`:

In [ ]:
FIVE = ["Gr_Liv_Area", "Year_Built", "Overall_Qual", "Total_Bsmt_SF", "Lot_Area"]
X5_train, X5_test, y5_train, y5_test = train_test_split(
    ames[FIVE], y, test_size=0.25, random_state=764)

for name, model in [("OLS", LinearRegression()), ("ridge", Ridge(alpha=10)),
                    ("lasso", Lasso(alpha=1000))]:
    scaled = pipeline(model, FIVE, [], scale=True).fit(X5_train, y5_train).predict(X5_test)
    raw = pipeline(model, FIVE, [], scale=False).fit(X5_train, y5_train).predict(X5_test)
    print(f"  {name:<6} largest change in any prediction when the scaler is removed: "
          f"${np.abs(scaled - raw).max():,.6f}")

Now change nothing about the houses — only the units of one column. `Lot_Area` is in
square feet. Put it in acres.

In [ ]:
for unit, factor in [("square feet", 1), ("acres", 1 / 43_560)]:
    X_unit = X5_train.copy()
    X_unit["Lot_Area"] = X_unit["Lot_Area"] * factor
    for scale in (True, False):
        lasso = pipeline(Lasso(alpha=1000), FIVE, [], scale=scale).fit(X_unit, y5_train)
        coef = lasso.named_steps["model"].coef_[FIVE.index("Lot_Area")]
        print(f"  Lot_Area in {unit:<12} scaler {'on ' if scale else 'off'}   "
              f"lasso coefficient {coef:>12.4f}")

**With the scaler, the lasso does not notice** — the same coefficient in either unit,
because it only ever sees Lot_Area in standard deviations.

**Without it, the lasso drops the column** the moment it is measured in acres. An acre is
43,560 square feet, so the coefficient has to be 43,560 times larger to mean the same
thing — and the lasso charges for size. It decided a lot's area does not matter because of
the units somebody typed.

⚠ **So the `StandardScaler` is no longer cosmetic, and it still goes inside the
pipeline.** The scaling statistics are estimated from data like everything else.

## Studio

Thirty minutes. The 36 numeric columns only, and **300 houses** — pick your own sample
by changing `764` below to any number you like.

1. **Watch the coefficients move.** Fit the lasso at every α in `ALPHAS` and plot each
   coefficient against α (log scale on the x-axis). Which column is the last one standing?
   How many are left at α = 3000? Then do the same with `Ridge` over the same grid — does
   any ridge coefficient ever reach exactly zero?
2. **Change the units.** Put `Lot_Area` in acres and refit the lasso at α = 3000, with the
   scaler and without it. Which pipeline noticed? Try one more column in units of your
   choice.
3. **Refit on ten different samples.** Draw ten different 300-house samples, fit the lasso
   at α = 3000 on each, and count for every column how many of the ten kept it. How many
   columns were kept **every** time?

⚠ Question 3 is the one to finish. Before you run it, write down how many of the columns
you expect to survive all ten.

In [ ]:
sample = ames.sample(300, random_state=764)       # <- your own number here
X_s, y_s = sample[NUMERIC], sample["SalePrice"]
ALPHAS = np.logspace(1, 5, 40)                    # 10 to 100,000

# pipeline(Lasso(alpha=a, max_iter=10_000), NUMERIC, [])  is the numeric-only lasso

In [ ]:
# YOUR CODE HERE — Question 1: coefficient paths

In [ ]:
# YOUR CODE HERE — Question 2: Lot_Area in acres, scaler on and off

In [ ]:
# YOUR CODE HERE — Question 3: ten samples, how often is each column kept?

## Compare

1. Which column was the last one standing on your sample? Did anyone get a different one?
2. Did anyone's pipeline keep `Lot_Area` in acres **without** the scaler?
3. How many columns survived all ten of your samples — and how many did you expect?

## Exit ticket

> The lasso kept about eighteen columns on your sample. Write one sentence about which
> features "matter" that you could defend, and one you would refuse to write.

**Answer in the cell below — two or three sentences — then save this notebook and upload
it to Canvas under "Studio notebook — Day 7". It is not graded. It is how I find out what
landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **Elastic net.** A penalty that is part ridge, part lasso — `ElasticNet` in
scikit-learn. It keeps the lasso's zeros but behaves better when columns are correlated,
which is exactly where the lasso's choices were least stable today. ESL §3.4 has the
geometry.

📗 **Post-selection inference.** There are honest ways to get p-values after a data-driven
choice of predictors. They are subtle, and they are the reason nobody should run the
Section 1 procedure and report its table.

🚫 **Stepwise selection by p-value.** Section 1. Every number it prints is correct for a
question you did not ask.